# Sheild — Version 2 CPU model training
Upload `Sheild_Training_V2.zip` when prompted. All data is independently generated synthetic data. V1 is frozen for comparison on the same new synthetic test set. Keep the official toolkit untouched. Run only when permitted by the original-work rules. No hidden evaluator results are produced here.


In [ ]:
# 1. CPU runtime; install training/inference dependency
import sys, subprocess, os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-crfsuite==0.9.12'])

In [ ]:
# 2. Upload V2 package; extract only the expected source and data.
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io, shutil
uploaded = files.upload()
name = 'Sheild_Training_V2.zip'
assert name in uploaded, f'Upload {name}'
work = Path('/content/sheild_training_v2')
work.mkdir(exist_ok=True)
expected = {'generate_dataset.py','model_features.py','model_runtime.py','train_models.py','requirements.txt','verify_package.py','data/manifest.json'}
expected |= {f'data/{kind}_{split}.jsonl' for kind in ('ner','linkage') for split in ('train','validation','test')}
expected |= {'baseline_v1/'+p for p in ('evaluate_v1.py','train_models.py','pii_crf.crfsuite','linkage_parameters.json','model_metadata.json','evaluation_metrics.json','model_features.py','model_runtime.py')}
with ZipFile(io.BytesIO(uploaded[name])) as z:
    for rel in sorted(expected):
        src = 'training/v2/' + rel
        assert src in z.namelist(), f'Missing {src}'
        p = work / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        with z.open(src) as a, p.open('wb') as b: shutil.copyfileobj(a,b)
del uploaded
os.chdir(work)
print('V2 training source and datasets extracted.')


In [ ]:
# 3. Check the dataset and shared runtime policy before training.
import json
subprocess.check_call([sys.executable,'verify_package.py'])


In [ ]:
# 4. Compare three CRF regularization settings on validation only; train the selected configuration.
# CPU is sufficient. This is a larger dataset; training time and memory vary by Colab runtime.
export_root = work / 'export'
model_dir = export_root / 'models' / 'v2'
if export_root.exists(): shutil.rmtree(export_root)
subprocess.check_call([sys.executable,'train_models.py','--data','data','--output',str(model_dir),'--iterations','150'])
# Frozen V1 is tested in a separate process using its original feature code.
subprocess.check_call([sys.executable,'baseline_v1/evaluate_v1.py','--data',str(work/'data'),'--output',str(model_dir/'baseline_v1_on_v2_test.json')])


In [ ]:
# 5. Compare measured metrics on identical test data; no tuning on test results.
import pandas as pd
metrics = json.loads((model_dir / 'evaluation_metrics.json').read_text())
baseline = json.loads((model_dir/'baseline_v1_on_v2_test.json').read_text())
summary=[]
for version,ner,link in [('V1',baseline['ner'],baseline['linkage']),('V2',metrics['ner']['test'],metrics['linkage']['test'])]:
    summary += [dict(version=version,model='CRF exact PII spans',**{k:ner['pii_exact_span_micro'][k] for k in ('precision','recall','f1')}),dict(version=version,model='Linkage candidate pairs',**{k:link[k] for k in ('precision','recall','f1')})]
display(pd.DataFrame(summary))
for version,ner in [('V1',baseline['ner']),('V2',metrics['ner']['test'])]:
    print(version,'per-language metrics'); display(pd.DataFrame(ner['per_language']).T)
    print(version,'per-label metrics'); display(pd.DataFrame(ner['per_label']).T)
    print('Protected overlap:',ner['protected_overlap_rate'])
print('Selected CRF configuration:',metrics['ner']['selected_config'])
print('Selected linkage threshold:',metrics['linkage']['threshold_selected_on_validation'])
print('Synthetic held-out results only, not official evaluator scores.')


In [ ]:
# 6. Reload V2 with its own feature contract; validate conservative decisions.
from model_runtime import CandidateModels
runtime = CandidateModels(model_dir)
assert isinstance(runtime.spans('मरीज उदाहरण नाम; आयु ४५ वर्ष; BP 120/80 mmHg.'), list)
assert not runtime.pair({'strong_id':'SYN-A'}, {'strong_id':'SYN-B'})['candidate_link']
assert not runtime.pair({'name':'Example Person','phone':'0000000000'}, {'name':'Example Person','phone':'0000000000'})['candidate_link']
print('V2 artifact reload and conservative checks passed.')


In [ ]:
# 7. Export versioned artifacts without overwriting V1.
from zipfile import ZIP_DEFLATED
required = ['pii_crf.crfsuite','linkage_parameters.json','evaluation_metrics.json','model_metadata.json','model_features.py','model_runtime.py','baseline_v1_on_v2_test.json']
assert all((model_dir/p).is_file() and (model_dir/p).stat().st_size > 0 for p in required)
out = Path('/content/Sheild_Trained_Models_V2.zip')
with ZipFile(out,'w',compression=ZIP_DEFLATED) as z:
    for p in required: z.write(model_dir/p, 'models/v2/'+p)
print('Exported:', out.name)
files.download(str(out))


## Next step
Extract `Sheild_Trained_Models_V2.zip` into the existing project root. Keep `models/` V1 files. Do not overwrite V1 with V2 feature code. Review the fresh comparison before choosing a model; do not treat synthetic metrics as final privacy proof. Append `python-crfsuite==0.9.12` to root requirements if missing. Integrate in the planned detection/linkage phases only, with clinical guards and independent verification.
